# Shingan AI - Colab Smoke Test (Experiment 001)

This notebook verifies that the Baseline CNN, Data Pipeline, and Training Loop run successfully on a Colab T4 GPU before moving on to full-scale training.
It mounts Google Drive to access the dataset without keeping heavy files in the GitHub repository.

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Setup GitHub Repository

In [ ]:
# Clone the repository (Update the URL to your actual repo)
!git clone https://github.com/<your-username>/<repo-name>.git
%cd <repo-name>

# Install requirements
!pip install -r requirements.txt

## 3. Verify Dataset Exists in Drive
Make sure your Drive has the structure:
`MyDrive/Shingan_AI/datasets/train/GT` and `NoisyLR`
`MyDrive/Shingan_AI/datasets/test/GT` and `NoisyLR`

In [ ]:
import os

train_dir = "/content/drive/MyDrive/Shingan_AI/datasets/train"
test_dir = "/content/drive/MyDrive/Shingan_AI/datasets/test"

print("Train directory contents:", os.listdir(train_dir))
print("Test directory contents:", os.listdir(test_dir))

## 4. Run the Smoke Test

We will run `train.py` but tell it to only use `16` samples for 2 epochs.

In [ ]:
import yaml

with open("configs/default.yaml", "r") as f:
    config = yaml.safe_load(f)

# Override for smoke test and Google Drive paths
config['training']['epochs'] = 2
config['data']['train_dir'] = train_dir
config['data']['val_dir'] = test_dir
config['data']['test_dir'] = test_dir
config['training']['save_dir'] = "/content/drive/MyDrive/Shingan_AI/checkpoints"

with open("configs/smoke_test.yaml", "w") as f:
    yaml.dump(config, f)

print("Created smoke_test.yaml configuration.")

In [ ]:
!python trainer/train.py --config configs/smoke_test.yaml --subset 16

## 5. Test Inference Engine
Verify the forward pass and output saving on a single image from Drive.

In [ ]:
!python scripts/infer.py \
    --input "/content/drive/MyDrive/Shingan_AI/datasets/test/NoisyLR/some_image_name.npy" \
    --output "/content/drive/MyDrive/Shingan_AI/results/output_test.npy" \
    --checkpoint "/content/drive/MyDrive/Shingan_AI/checkpoints/best_model.pth" \
    --config configs/smoke_test.yaml